# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *70* |
| **Integrantes** | *Isabela Cuartas Vence · Juan Camilo Gomez Murillo* |
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 23 de agosto |
| **🎥 Enlace al video** | *https://drive.google.com/drive/folders/19hTos1h6VIlHoUuDAMm3DBIeHMqf_4qQ?usp=sharing* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.*

Wanderbricks es una plataforma de alquiler de vacaciones donde los anfitriones pueden listar sus propiedades y los usuarios pueden hacer reservas que luego pueden ser modificadas, confirmadas o canceladas. El núcleo del problema empresarial no se limita a conocer la cantidad de reservas, sino a comprender su evolución con qué frecuencia se producen cambios de estado, cuánto tiempo pasa desde la creación de una reserva hasta su actualización, y si este comportamiento difiere según el país del usuario o de la propiedad. 

La tabla booking_updates es la que hace visible esta dinámica temporal, mientras que users y countries permiten segmentar ese comportamiento por perfil de usuario y por geografía. Esto es relevante ya que el equipo de operaciones y el equipo comercial de Wanderbricks requieren decisiones fundamentadas en datos, no en suposiciones identificar mercados con alta tasa de cancelación para modificar políticas de reembolso, y detectar si ciertos comentarios de los usuarios provocan más cambios de última hora y permiten anticipar picos de demanda por región. Sin una base de datos que conserve el historial de cambios de cada reserva , estas preguntas no se pueden responder, porque la tabla bookings por sí sola solo muestra una fotografía del presente, no la trayectoria de cada reserva.
 
Esta base de datos debe proporcionar respuestas a preguntas como: ¿cuál es la tasa de cancelación o modificación de reservas en cada país?, ¿cuánto tiempo, en promedio, transcurre entre la creación de una reserva y su primera actualización?, ¿qué países tienen la mayor cantidad de usuarios activos y cómo se relaciona esto con el número de reservas confirmadas?, ¿hay patrones de comportamiento que ayuden a predecir cancelaciones?, y ¿cómo ha variado el volumen de reservas y actualizaciones a lo largo del tiempo según la región geográfica?

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

In [0]:
# TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)
from pyspark.sql import functions as F

tablas = ["users", "countries", "bookings", "booking_updates"]

for t in tablas:
    print(f"\n--- Nulos en {t} ---")
    df = spark.table(f"samples.wanderbricks.{t}")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

#Cardinalidades (columna por columna, para cada tabla):

for t in tablas:
    print(f"\n--- Cardinalidad en {t} ---")
    df = spark.table(f"samples.wanderbricks.{t}")
    total = df.count()
    df.select([F.countDistinct(F.col(c)).alias(c) for c in df.columns]).display()
    print(f"(Total de filas para comparar: {total:,})")

In [0]:
# Esquema y tipos de cada tabla
for t in tablas:
    print(f"\n--- Esquema de {t} ---")
    spark.table(f"samples.wanderbricks.{t}").printSchema()

Relaciones entre tablas: 
bookings.user_id → users.user_id (N:1): cada reserva pertenece a un usuario.

booking_updates.booking_id → bookings.booking_id (N:1): cada reserva puede tener cero o varias actualizaciones (83,068 actualizaciones sobre 72,247 reservas).

users.country → countries.country (N:1): la relación geográfica se hace por nombre de país, no por country_code, aunque ambas columnas existen en countries.

Esto es un hallazgo de calidad de datos a tener en cuenta: exige coincidencia exacta de string (sensible a mayúsculas/tildes/espacios), lo cual valida antes del join.

Observación sobre booking_updates
El esquema de booking_updates replica casi por completo el de bookings (check_in, check_out, guests_count, status, total_amount), agregando booking_update_id y updated_at. Esto confirma que no es una tabla relacional tradicional de "detalle", sino un log de cambios, cada fila representa el 
estado de una reserva en un momento dado. Este patrón encaja naturalmente con la capacidad de time travel de Delta Lake, que exploraremos en la sección 4.5.

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*



### Comparación de paradigmas atada al caso Wanderbricks

| Criterio del caso | Relacional (PostgreSQL/MySQL) | NoSQL documental (MongoDB) | Lakehouse (Delta Lake) |
|---|---|---|---|
| **Volumen y tipo de carga** | Cómodo hasta ~500K filas (Bookkeeping, 2019), pero pensado para OLTP fila por fila, no para escaneos analíticos masivos sobre 124,509 usuarios / 72,246 reservas. | Escala horizontalmente bien, pero su fuerza es la lectura/escritura de documentos individuales, no `GROUP BY` sobre millones de filas como los de nuestras 5 consultas analíticas. | Optimizado para lectura columnar sobre grandes volúmenes (Armbrust et al., 2021); nuestras consultas (Q1, Q4) hacen `JOIN` + `GROUP BY` sobre las 4 tablas completas sin degradación notable. |
| **Relaciones entre entidades** | Fuerte: `bookings.user_id → users.user_id` y `booking_updates.booking_id → bookings.booking_id` son relaciones N:1 naturales para claves foráneas e integridad referencial declarada (Codd, 1970). | Débil para este caso: modelar 4 colecciones relacionadas exige *embedding* o referencias manuales sin integridad garantizada por el motor (MongoDB, 2024). | Soporta joins tipo SQL igual que el modelo relacional (usados en las 5 consultas), sin forzar integridad referencial estricta a nivel de motor — trade-off que asumimos y mitigamos con reglas de negocio explícitas en la capa plata (sección 4.3). |
| **Datos semiestructurados (1:N embebido)** | No nativo: `booking_updates` (83,068 filas, más filas que `bookings`) requeriría una tabla aparte y JOIN obligatorio para reconstruir el historial de cada reserva. | Fuerte candidato natural: un documento por reserva con un array embebido de actualizaciones (MongoDB, 2024). | Igual de capaz que el documental: `silver_bookings_anidado` (sección 4.4) demuestra que Delta Lake soporta `array<struct>` de forma nativa, sin sacrificar la capacidad de hacer `JOIN`/`GROUP BY` relacional sobre el resto de los datos. |
| **Auditoría y versionado histórico** | Requiere tablas de auditoría manuales (triggers, tablas `_history`) construidas a mano. | No tiene versionado de esquema/tabla nativo; se gestiona a nivel de aplicación. | Nativo vía transaction log (`DESCRIBE HISTORY`, sección 4.5): se demostró consultar la versión 0 (1,377 `pending`) vs. la versión 1 (mismas filas como `expired`) sin tablas de auditoría adicionales (Armbrust et al., 2020). |
| **Evolución de esquema** | Requiere `ALTER TABLE` explícito y migraciones coordinadas; alto costo si el equipo agrega columnas con frecuencia. | Flexible por naturaleza (sin esquema fijo), pero eso también dificulta garantizar calidad de datos consistente. | `mergeSchema=true` permitió agregar `duracion_noches` a una tabla con 72,246 filas ya escrita, sin migración ni downtime (sección 4.5), conservando el tipado explícito que sí exigimos en bronce/plata. |
| **Consistencia transaccional (ACID)** | Fuerte, pero pensada para transacciones OLTP pequeñas, no para un `UPDATE` masivo sobre 1,377 filas como el de la sección 4.5. | Consistencia eventual en muchas configuraciones distribuidas; no es el foco del motor. | ACID garantizado incluso en operaciones de gran volumen sobre archivos Parquet (Armbrust et al., 2020); se evidenció con el `UPDATE` atómico de reservas `expired`. |
| **Costo/complejidad operativa** | Bajo para volúmenes pequeños, pero exige separar un data warehouse aparte si se necesita analítica pesada (arquitectura de dos sistemas). | Bajo para el caso documental puro, pero exigiría un sistema analítico adicional para las consultas tipo Q1/Q4. | Un solo sistema para ingesta, transformación (bronce/plata) y analítica, evitando duplicar infraestructura (Armbrust et al., 2021). |

### Decisión

Se elige **lakehouse (Delta Lake)** sobre un motor relacional puro porque el caso combina dos necesidades que el modelo relacional separa en dos sistemas distintos: manejo eficiente de datos semiestructurados (el historial de `booking_updates` como array anidado) y analítica agregada de volumen medio-alto (124,509 usuarios, 5 consultas con `JOIN` y `GROUP BY`) — sin sacrificar ACID ni relaciones tipo clave foránea, que sí necesitamos para mantener la integridad entre `bookings`, `users` y `countries`.

Se descarta **NoSQL documental puro** porque, si bien modela mejor el historial anidado de una reserva, nuestras preguntas de negocio (tasa de cancelación por país, volumen por continente y mes) requieren agregaciones y joins multi-colección que MongoDB resuelve de forma menos natural que un motor con soporte SQL nativo, y perderíamos el versionado transaccional (time travel) que sí usamos activamente en la sección 4.5.

Se descarta el modelo **relacional puro** porque, aunque es el más natural para las relaciones `users`–`bookings`–`countries`, no soporta de forma nativa la estructura anidada de `booking_updates` sin normalizarla en una tabla aparte (lo cual no es un problema en sí, pero no aprovecha la ventaja de Parquet/Delta para este patrón), y no ofrece versionado de tabla nativo — tendríamos que construir manualmente lo que `DESCRIBE HISTORY` nos dio gratis.

### Referencias (APA 7)

Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. (2021). *Lakehouse: A new generation of open source and proprietary data platforms*. Proceedings of CIDR 2021. https://www.cidrdb.org/cidr2021/papers/cidr2021_paper17.pdf

Armbrust, M., Das, T., Sun, L., Yavuz, B., Zhu, S., Murthy, M., Torres, J., van Hovell, H., Ionescu, A., Łuszczak, A., Świtakowski, M., Szafrański, M., Li, X., Ueshin, T., Mokhtar, M., Boncz, P., Ghodsi, A., Paranjpye, S., Senster, P., & Zaharia, M. (2020). Delta Lake: High-performance ACID table storage over cloud object stores. *Proceedings of the VLDB Endowment, 13*(12), 3411–3424. https://doi.org/10.14778/3415478.3415560

Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM, 13*(6), 377–387. https://doi.org/10.1145/362384.362685

MongoDB, Inc. (2024). *MongoDB documentation: Data modeling*. MongoDB. https://www.mongodb.com/docs/manual/core/data-modeling-introduction/

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo70"   # TODO: reemplazar NN
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# TODO: ingerir las tablas del caso a la capa bronce, con esquema explícito
from pyspark.sql import functions as F

FUENTE = "samples.wanderbricks"

def ingerir_bronce(nombre_tabla, esquema_explicito, nombre_bronce=None):
    nombre_bronce = nombre_bronce or f"bronze_{nombre_tabla}"
    df_raw = spark.table(f"{FUENTE}.{nombre_tabla}")

    exprs = [F.col(c).cast(t).alias(c) for c, t in esquema_explicito.items()]
    df_tipado = df_raw.select(*exprs)

    df_bronce = (df_tipado
        .withColumn("_source_table", F.lit(f"{FUENTE}.{nombre_tabla}"))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (df_bronce.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.{nombre_bronce}"))

    print(f"✔ {nombre_bronce}: {df_bronce.count():,} filas ingeridas")


# --- Esquemas explícitos (nombres reales, confirmados por printSchema) ---

esquema_users = {
    "user_id":      "bigint",
    "email":        "string",
    "name":         "string",
    "country":      "string",
    "user_type":    "string",
    "created_at":   "timestamp",
    "is_business":  "boolean",
    "company_name": "string",
}

esquema_countries = {
    "country":      "string",
    "country_code": "string",
    "continent":    "string",
}

esquema_bookings = {
    "booking_id":   "bigint",
    "user_id":      "bigint",
    "property_id":  "bigint",
    "check_in":     "date",
    "check_out":    "date",
    "guests_count": "int",
    "total_amount": "float",
    "status":       "string",
    "created_at":   "timestamp",
    "updated_at":   "timestamp",
}

esquema_booking_updates = {
    "booking_update_id": "bigint",
    "booking_id":         "bigint",
    "user_id":            "bigint",
    "property_id":        "bigint",
    "check_in":           "date",
    "check_out":          "date",
    "guests_count":       "int",
    "total_amount":       "float",
    "status":             "string",
    "created_at":         "timestamp",
    "updated_at":         "timestamp",
}

# --- Ejecutar la ingesta ---
ingerir_bronce("users", esquema_users)
ingerir_bronce("countries", esquema_countries)
ingerir_bronce("bookings", esquema_bookings)
ingerir_bronce("booking_updates", esquema_booking_updates)

### 4.3 Capa plata — datos limpios y tipados

### Reglas de negocio aplicadas en capa plata

- **silver_bookings**: se descartan reservas con `check_out <= check_in` (fechas 
  inconsistentes) y `total_amount < 0` (dato inválido).
- **silver_users / silver_countries**: se normaliza el nombre de país (trim + 
  minúsculas) para garantizar el join correcto, dado que la relación entre estas 
  tablas se hace por nombre de país y no por código ISO.
- **silver_bookings_estado_actual**: se combina cada reserva con su actualización 
  más reciente (según `updated_at`) usando `ROW_NUMBER()`. Si una reserva nunca 
  fue actualizada, se conserva su estado original. Esto responde a la pregunta de 
  negocio "¿cuál es el estado vigente de cada reserva?", sin perder el historial 
  completo, que se conserva por separado en `silver_booking_updates`.

In [0]:
# TODO: limpieza, tipado y reglas de negocio

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -----------------------------------------------------------------
# 4.3 Capa plata — datos limpios y tipados
# -----------------------------------------------------------------

# --- silver_users: limpieza básica + normalización de country para el join ---
silver_users = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_users")
    .withColumn("country_clean", F.trim(F.lower(F.col("country"))))
    .filter(F.col("user_id").isNotNull())            # regla de negocio: sin user_id no sirve
    .dropDuplicates(["user_id"])                       # por si hay duplicados exactos
)

# --- silver_countries: misma normalización para que el join calce ---
silver_countries = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_countries")
    .withColumn("country_clean", F.trim(F.lower(F.col("country"))))
    .dropDuplicates(["country"])
)

# --- silver_bookings: reglas de negocio básicas de validez ---
silver_bookings = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings")
    .filter(F.col("check_out") > F.col("check_in"))   # una reserva debe tener fechas coherentes
    .filter(F.col("total_amount") >= 0)                 # montos negativos no tienen sentido
)

# --- silver_booking_updates: historial completo, tipado, sin cambios de negocio ---
silver_booking_updates = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_booking_updates")

# --- silver_bookings_estado_actual: bookings + último estado conocido ---
w = Window.partitionBy("booking_id").orderBy(F.col("updated_at").desc())

ultimo_update = (silver_booking_updates
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumnRenamed("status", "status_actualizado")
    .withColumnRenamed("total_amount", "total_amount_actualizado")
    .withColumnRenamed("updated_at", "ultima_actualizacion")
    .select("booking_id", "status_actualizado", "total_amount_actualizado", "ultima_actualizacion")
)

silver_bookings_estado_actual = (silver_bookings
    .join(ultimo_update, on="booking_id", how="left")
    # si nunca hubo actualización, el estado vigente es el original de bookings
    .withColumn("status_vigente",
        F.coalesce(F.col("status_actualizado"), F.col("status")))
    .withColumn("monto_vigente",
        F.coalesce(F.col("total_amount_actualizado"), F.col("total_amount")))
)

# --- Escribir todas las tablas plata ---
tablas_plata = {
    "silver_users": silver_users,
    "silver_countries": silver_countries,
    "silver_bookings": silver_bookings,
    "silver_booking_updates": silver_booking_updates,
    "silver_bookings_estado_actual": silver_bookings_estado_actual,
}

for nombre, df in tablas_plata.items():
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.{nombre}"))
    print(f"✔ {nombre}: {df.count():,} filas")

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

### Manejo de datos semiestructurados

Las tablas fuente de este proyecto son planas. Para cumplir con el manejo de 
estructuras anidadas de forma que aporte valor real (no artificial), construimos 
`silver_bookings_anidado`: cada reserva incluye un array de structs con su 
historial completo de actualizaciones (`historial_actualizaciones`), en vez de 
requerir un join contra `booking_updates` para conocer ese historial. Esto 
demuestra una ventaja práctica del formato Delta/Parquet frente al modelo 
relacional estricto: una relación 1:N puede vivir dentro de una sola fila, 
lo cual simplifica ciertas consultas analíticas (por ejemplo, "reservas con 
más de 3 modificaciones") sin sacrificar la posibilidad de aplanar (`explode`) 
cuando se necesite trabajar update por update.

In [0]:
# TODO: leer y aplanar estructuras anidadas
from pyspark.sql import functions as F

# -----------------------------------------------------------------
# 4.4 Datos semiestructurados
# -----------------------------------------------------------------
# Construimos una tabla donde cada reserva anida, en un array de structs,
# el historial completo de sus actualizaciones. Esto modela de forma nativa
# la relación 1:N booking -> booking_updates, sin necesitar un join en 
# tiempo de consulta.

updates_agrupados = (silver_booking_updates
    .groupBy("booking_id")
    .agg(
        F.collect_list(
            F.struct(
                F.col("booking_update_id"),
                F.col("status").alias("nuevo_status"),
                F.col("total_amount").alias("nuevo_monto"),
                F.col("check_in").alias("nuevo_check_in"),
                F.col("check_out").alias("nuevo_check_out"),
                F.col("updated_at")
            )
        ).alias("historial_actualizaciones")
    )
)

silver_bookings_anidado = (silver_bookings
    .join(updates_agrupados, on="booking_id", how="left")
    .withColumn(
        "num_actualizaciones",
        F.when(F.col("historial_actualizaciones").isNotNull(),
               F.size(F.col("historial_actualizaciones"))).otherwise(0)
    )
)

(silver_bookings_anidado.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings_anidado"))

print(f"✔ silver_bookings_anidado: {silver_bookings_anidado.count():,} filas")

# Verificación visual de la estructura anidada
silver_bookings_anidado.select(
    "booking_id", "status", "num_actualizaciones", "historial_actualizaciones"
).filter(F.col("num_actualizaciones") > 1).show(5, truncate=False)

silver_bookings_anidado.printSchema()

### Hallazgo relevante

Se observó que `bookings.status` no siempre refleja el estado más reciente de 
una reserva. Por ejemplo, la reserva `2392` tiene `status = pending` en la tabla 
original, pero su actualización más reciente (22:45:23) indica `confirmed`. Esto 
valida la necesidad de `silver_bookings_estado_actual`, que resuelve el estado 
vigente combinando `bookings` con el último registro de `booking_updates` según 
`updated_at`.

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
# ACID — una operación que modifique datos
# TODO

# --- ACID: operación atómica sobre silver_bookings_estado_actual ---
from pyspark.sql import functions as F

# Contamos cuántas reservas "pending" con check_in ya pasado hay antes del cambio
pendientes_vencidas = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual
    WHERE status_vigente = 'pending' AND check_in < current_date()
""")
pendientes_vencidas.show()

# UPDATE atómico: Delta garantiza que esta operación se aplica completa o no se aplica
spark.sql(f"""
    UPDATE {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual
    SET status_vigente = 'expired'
    WHERE status_vigente = 'pending' AND check_in < current_date()
""")

# Verificamos el resultado
spark.sql(f"""
    SELECT status_vigente, COUNT(*) AS total
    FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual
    GROUP BY status_vigente
    ORDER BY total DESC
""").show()

### Atomicidad (ACID)

Se ejecutó un `UPDATE` sobre `silver_bookings_estado_actual` para marcar como 
`expired` las reservas que seguían en estado `pending` pese a tener el `check_in` 
en el pasado (regla de negocio: una reserva pendiente después de su fecha de 
check-in ya no es válida). Antes del `UPDATE` se detectaron 1,377 reservas en 
esta condición; después de aplicarlo, el conteo por estado confirma que las 
1,377 pasaron a `expired` sin afectar las demás categorías (total de 72,246 
filas se mantiene). Delta Lake garantiza que esta operación se aplica de forma 
atómica: o se actualizan todas las filas que cumplen la condición, o ninguna, 
sin dejar la tabla en un estado intermedio inconsistente.

In [0]:
# Time travel
# display(spark.sql(f"DESCRIBE HISTORY {TABLA}"))
# TODO: consultar una versión anterior y comparar

# --- Time Travel ---

# Vemos el historial de versiones de la tabla
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual"))

### Time Travel

Se consultó `DESCRIBE HISTORY` sobre `silver_bookings_estado_actual`, confirmando 
dos versiones: la versión 0 (creación de la tabla, 72,246 filas) y la versión 1 
(el `UPDATE` de la sección anterior, que modificó 1,377 filas según 
`numUpdatedRows` en las métricas de la operación).

Al consultar la versión 0 con `VERSION AS OF`, se observa `status_vigente = pending` 
con 1,377 registros y ningún registro `expired`. Al consultar la versión actual, 
esas mismas 1,377 filas aparecen como `expired` y `pending` desaparece del todo. 
Esto demuestra que Delta Lake conserva el estado histórico completo de la tabla 
y permite auditar o revertir cambios sin perder información, algo que una 
sobrescritura tradicional (`UPDATE` en una base relacional sin versionado) no 
ofrecería de forma nativa.

In [0]:
# --- Comparar versión anterior (antes del UPDATE) vs. versión actual ---
from pyspark.sql import functions as F

# Identificar dinámicamente la versión más reciente de WRITE (antes del UPDATE)
historial = spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
version_antes = (historial
    .filter(F.col("operation") == "CREATE OR REPLACE TABLE AS SELECT")
    .select(F.max("version"))
    .first()[0])

df_antes = spark.read.format("delta") \
    .option("versionAsOf", version_antes) \
    .table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")

print(f"--- Versión {version_antes} (antes del UPDATE) ---")
df_antes.groupBy("status_vigente").count().orderBy(F.col("count").desc()).show()

df_version_actual = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")

print("--- Versión actual (después del UPDATE) ---")
df_version_actual.groupBy("status_vigente").count().orderBy(F.col("count").desc()).show()

In [0]:
# Evolución de esquema con mergeSchema
# TODO

# --- Evolución de esquema ---

# Esquema ANTES del cambio
print("--- Esquema antes ---")
spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual").printSchema()

# Agregamos una columna nueva calculada
df_con_nueva_columna = (spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
    .withColumn("duracion_noches", F.datediff(F.col("check_out"), F.col("check_in")))
)

# Escribimos con mergeSchema=true para evolucionar el esquema sin recrear la tabla
(df_con_nueva_columna.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual"))

# Esquema DESPUÉS del cambio
print("--- Esquema después ---")
spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual").printSchema()

# Confirmamos con DESCRIBE HISTORY que quedó registrada una nueva versión
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual"))

### Evolución de esquema

Se agregó la columna calculada `duracion_noches` (días entre `check_in` y 
`check_out`) a `silver_bookings_estado_actual` usando `mergeSchema=true`, sin 
necesidad de recrear la tabla ni definir de antemano todas las columnas futuras. 
El `printSchema()` antes y después confirma que la nueva columna se incorporó 
al final del esquema existente, preservando las 17 columnas anteriores. Esta 
capacidad es relevante para el caso de negocio: permite que el equipo de 
analítica agregue métricas derivadas de forma incremental a medida que surgen 
nuevas preguntas, sin coordinar una migración de esquema como se requeriría en 
una base de datos relacional con tipado estricto desde el DDL.

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

Version SQL

¿En que paises se concentran mas las cancelaciones o reservas vencidas sin gestionar?
Esto le sirve al negocio para detectar mercados con politicas de rembolso o comunicacion deficientes

In [0]:
# Consulta 1 — pregunta que responde:
# TODO

spark.sql(f"""
SELECT
    c.country,
    c.continent,
    COUNT(*) AS total_reservas,
    SUM(CASE WHEN b.status_vigente = 'cancelled' THEN 1 ELSE 0 END) AS canceladas,
    SUM(CASE WHEN b.status_vigente = 'expired' THEN 1 ELSE 0 END) AS vencidas,
    ROUND(100.0 * SUM(CASE WHEN b.status_vigente IN ('cancelled','expired') THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_problema_pct
FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual b
JOIN {CATALOGO}.{ESQUEMA}.silver_users u
    ON b.user_id = u.user_id
JOIN {CATALOGO}.{ESQUEMA}.silver_countries c
    ON TRIM(LOWER(u.country)) = TRIM(LOWER(c.country))
GROUP BY c.country, c.continent
HAVING COUNT(*) >= 30
ORDER BY tasa_problema_pct DESC
LIMIT 15
""").show(15, truncate=False)

Versión PySpark


In [0]:
from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_users")
df_countries = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_countries")

resultado_q1 = (df_bookings
    .join(df_users, on="user_id")
    .join(
        df_countries,
        on=F.trim(F.lower(df_users["country"])) == F.trim(F.lower(df_countries["country"])),
        how="inner"
    )
    .groupBy(df_countries["country"], df_countries["continent"])
    .agg(
        F.count("*").alias("total_reservas"),
        F.sum(F.when(F.col("status_vigente") == "cancelled", 1).otherwise(0)).alias("canceladas"),
        F.sum(F.when(F.col("status_vigente") == "expired", 1).otherwise(0)).alias("vencidas"),
        F.round(
            100.0 * F.sum(F.when(F.col("status_vigente").isin("cancelled", "expired"), 1).otherwise(0))
            / F.count("*"), 2
        ).alias("tasa_problema_pct")
    )
    .filter(F.col("total_reservas") >= 30)
    .orderBy(F.col("tasa_problema_pct").desc())
)

resultado_q1.show(15, truncate=False)

**Consulta 1 — Tasa de cancelación/expiración por país**

Los países con mayor proporción de reservas problemáticas (canceladas + vencidas) 
no muestran un patrón geográfico claro: aparecen países de Oceanía (Nueva Zelanda, 
53.3%), Asia (Singapur, 52.9%), Europa (Portugal, 49.4%) y África (Malawi, 49.3%) 
en las primeras posiciones. Sin embargo, se observa que la tasa de cancelación 
general del dataset ronda el 39% (28,287 de 72,246 reservas), por lo que incluso 
el "mejor" país de este top 15 está por encima del promedio general, sugiriendo 
que el problema de cancelación es transversal y no aislado a mercados específicos. 
Vale la pena una segunda mirada: con datos simulados, es posible que esta variable 
no tenga una relación causal real con el país, y que la tasa de cancelación 
responda más a otros factores (tipo de propiedad, anticipación de la reserva, etc.)

Version SQL

¿Cuanto tarda en promedio un usuario o el sistema en hacer el primer cambio sobre una reserva recien creada?
Esto ayuda a dimensionar si el proceso de confirmacion es lento y a priorizar automatizacion

In [0]:
spark.sql(f"""
WITH primera_actualizacion AS (
    SELECT
        booking_id,
        MIN(updated_at) AS primera_actualizacion
    FROM {CATALOGO}.{ESQUEMA}.silver_booking_updates
    GROUP BY booking_id
)
SELECT
    ROUND(AVG(TIMESTAMPDIFF(HOUR, b.created_at, p.primera_actualizacion)), 2) AS horas_promedio,
    ROUND(MIN(TIMESTAMPDIFF(HOUR, b.created_at, p.primera_actualizacion)), 2) AS horas_min,
    ROUND(MAX(TIMESTAMPDIFF(HOUR, b.created_at, p.primera_actualizacion)), 2) AS horas_max,
    COUNT(*) AS reservas_con_actualizacion
FROM {CATALOGO}.{ESQUEMA}.silver_bookings b
JOIN primera_actualizacion p
    ON b.booking_id = p.booking_id
""").show(truncate=False)

Versión PySpark

In [0]:
from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")
df_updates = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_booking_updates")

primera_actualizacion = (df_updates
    .groupBy("booking_id")
    .agg(F.min("updated_at").alias("primera_actualizacion"))
)

resultado_q2 = (df_bookings
    .join(primera_actualizacion, on="booking_id")
    .withColumn(
        "horas_hasta_actualizacion",
        (F.col("primera_actualizacion").cast("long") - F.col("created_at").cast("long")) / 3600
    )
    .agg(
        F.round(F.avg("horas_hasta_actualizacion"), 2).alias("horas_promedio"),
        F.round(F.min("horas_hasta_actualizacion"), 2).alias("horas_min"),
        F.round(F.max("horas_hasta_actualizacion"), 2).alias("horas_max"),
        F.count("*").alias("reservas_con_actualizacion")
    )
)

resultado_q2.show(truncate=False)

**Consulta 2 — Tiempo entre creación de reserva y su primera actualización**

De las 72,246 reservas, 47,705 (66%) tuvieron al menos una actualización. El 
tiempo promedio hasta la primera actualización es de 1,552 horas (~64.7 días), 
con un máximo de 20,258 horas (~844 días, más de 2 años). Este promedio es 
sorprendentemente alto para lo que se esperaría de un flujo real de confirmación 
de reservas (que normalmente ocurre en horas o pocos días). Esto sugiere que las 
fechas de `updated_at` en `booking_updates` no siguen necesariamente una relación 
temporal estrecha con `created_at` de `bookings` — posiblemente por tratarse de 
datos generados sintéticamente sin una regla de coherencia temporal estricta 
entre ambas tablas. Es un hallazgo de calidad de datos relevante para las 
conclusiones: en un caso real, este dato ameritaría investigar si el campo 
`created_at` de `booking_updates` fue mal poblado, en vez de asumir directamente 
que refleja demoras operativas reales.

Version SQL 

¿Que tan comun es que una reserva se modifique varias veces y esto se relaciona con que termine cancelada? 
Ayuda a decidir si conviene si poner friccion (costos, limites) a las modificaciones repetidas

In [0]:
spark.sql(f"""
SELECT
    a.num_actualizaciones,
    COUNT(*) AS cantidad_reservas,
    ROUND(100.0 * SUM(CASE WHEN e.status_vigente = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_canceladas,
    ROUND(100.0 * SUM(CASE WHEN e.status_vigente = 'expired' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_vencidas
FROM {CATALOGO}.{ESQUEMA}.silver_bookings_anidado a
JOIN {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual e
    ON a.booking_id = e.booking_id
GROUP BY a.num_actualizaciones
ORDER BY a.num_actualizaciones
""").show(20, truncate=False)

Versión PySpark

In [0]:
from pyspark.sql import functions as F

df_anidado = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_anidado")
df_estado_actual = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")

resultado_q3 = (df_anidado
    .join(df_estado_actual.select("booking_id", "status_vigente"), on="booking_id")
    .groupBy("num_actualizaciones")
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(100.0 * F.sum(F.when(F.col("status_vigente") == "cancelled", 1).otherwise(0)) / F.count("*"), 2).alias("pct_canceladas"),
        F.round(100.0 * F.sum(F.when(F.col("status_vigente") == "expired", 1).otherwise(0)) / F.count("*"), 2).alias("pct_vencidas")
    )
    .orderBy("num_actualizaciones")
)

resultado_q3.show(20, truncate=False)

**Consulta 3 — Reservas con más modificaciones y su relación con cancelación**

Se corrigió la consulta inicial: usar `bookings.status` (el estado original, sin 
actualizar) distorsionaba el resultado, mostrando 0% de cancelación en cualquier 
reserva con al menos una actualización — lo cual solo refleja que el estado 
`cancelled` casi siempre se define en `booking_updates`, no en el registro 
original. Usando `status_vigente` (estado real y actualizado) se observa un 
patrón claro y con sentido de negocio: a mayor número de actualizaciones, **menor** 
la tasa de cancelación (62.3% con 0 actualizaciones → 21.8% con 3 actualizaciones), 
pero **mayor** la tasa de reservas vencidas (`expired`) (0.37% → 9.12% en el mismo 
rango). Esto sugiere que las reservas que nunca se tocan tienden a cancelarse 
directamente, mientras que las que se modifican varias veces tienden a quedar 
"atascadas" en un limbo de pendiente hasta vencerse, en vez de resolverse. Esto 
es información accionable: el negocio podría implementar una alerta automática 
para reservas con 3+ modificaciones que no llegan a `confirmed`, antes de que 
lleguen a vencerse.

Version SQl
¿Como se distribuye la demanda en el tiempo y por region? 
Sirve para planear capacidad operativa y campañas de marketing y estacionales por continente

In [0]:
spark.sql(f"""
SELECT
    DATE_FORMAT(b.created_at, 'yyyy-MM') AS mes,
    c.continent,
    COUNT(*) AS total_reservas
FROM {CATALOGO}.{ESQUEMA}.silver_bookings b
JOIN {CATALOGO}.{ESQUEMA}.silver_users u
    ON b.user_id = u.user_id
JOIN {CATALOGO}.{ESQUEMA}.silver_countries c
    ON TRIM(LOWER(u.country)) = TRIM(LOWER(c.country))
GROUP BY DATE_FORMAT(b.created_at, 'yyyy-MM'), c.continent
ORDER BY mes, c.continent
""").show(30, truncate=False)

Versión PySpark

In [0]:
from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_users")
df_countries = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_countries")

resultado_q4 = (df_bookings
    .join(df_users, on="user_id")
    .join(
        df_countries,
        on=F.trim(F.lower(df_users["country"])) == F.trim(F.lower(df_countries["country"])),
        how="inner"
    )
    .withColumn("mes", F.date_format(df_bookings["created_at"], "yyyy-MM"))  # <- especificamos la tabla de origen
    .groupBy("mes", df_countries["continent"])
    .agg(F.count("*").alias("total_reservas"))
    .orderBy("mes", "continent")
)

resultado_q4.show(30, truncate=False)

**Consulta 4 — Volumen de reservas por mes y continente**

Se observa una tendencia de crecimiento sostenido en el volumen de reservas desde 
finales de 2022 (pocas reservas por mes) hasta mediados de 2023, con Asia y África 
como los continentes con mayor volumen en el periodo inicial. También se detectó 
un valor de calidad de datos a documentar: aparece `"Europe/Asia"` como valor de 
`continent` (probablemente correspondiente a un país transcontinental como Turquía 
o Rusia en la tabla `countries`), lo cual no es necesariamente un error, pero debe 
tratarse como una categoría aparte al segmentar por continente, y se recomienda 
revisar la tabla de dimensión `countries` para decidir si conviene reasignarlo a 
un continente único según la regla de negocio que se prefiera.

Version SQL

¿los usuarios de tipo empresa (is_business = true) generan reservas de mayor valor o duración que los individuales? Esto ayuda a decidir si vale la pena una estrategia comercial diferenciada (por ejemplo, tarifas corporativas o soporte dedicado).

In [0]:
spark.sql(f"""
SELECT
    u.is_business,
    COUNT(*) AS total_reservas,
    ROUND(AVG(b.monto_vigente), 2) AS monto_promedio,
    ROUND(AVG(b.duracion_noches), 2) AS noches_promedio,
    ROUND(AVG(b.guests_count), 2) AS huespedes_promedio
FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual b
JOIN {CATALOGO}.{ESQUEMA}.silver_users u
    ON b.user_id = u.user_id
GROUP BY u.is_business
""").show(truncate=False)

Versión PySpark

In [0]:
from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_users")

resultado_q5 = (df_bookings
    .join(df_users, on="user_id")
    .groupBy("is_business")
    .agg(
        F.count("*").alias("total_reservas"),
        F.round(F.avg("monto_vigente"), 2).alias("monto_promedio"),
        F.round(F.avg("duracion_noches"), 2).alias("noches_promedio"),
        F.round(F.avg("guests_count"), 2).alias("huespedes_promedio")
    )
)

resultado_q5.show(truncate=False)

**Consulta 5 — Usuarios business vs. individuales: monto y duración de reservas**

Los resultados muestran prácticamente **ninguna diferencia** entre usuarios 
business e individuales: monto promedio casi idéntico ($557.19 vs. $555.23), 
misma duración promedio (~3.75 noches) y mismo número de huéspedes (~1.8). Esto 
contradice la hipótesis inicial de que los usuarios empresariales generarían 
reservas de mayor valor. Una posible explicación es que `is_business` en este 
dataset distingue el *tipo de cuenta* del usuario, pero no necesariamente el 
*propósito* de cada reserva individual (un usuario de tipo "business" podría 
seguir reservando alojamientos vacacionales personales). Esto es un hallazgo de 
negocio válido en sí mismo: **no hay evidencia en estos datos para justificar una 
estrategia comercial diferenciada por tipo de cuenta**, salvo que se investigue 
más a fondo qué representa realmente ese campo.

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

**Volumen procesado**

Se ingirieron 4 tablas fuente de `samples.wanderbricks` hacia la capa bronce sin 
pérdida de filas (124,509 usuarios, 168 países, 72,247 reservas, 83,068 
actualizaciones). En la capa plata se aplicaron reglas de negocio que descartaron 
1 fila de `bookings` por inconsistencia de fechas (`check_out <= check_in`), 
quedando 72,246 reservas válidas como base para el resto del análisis.

### Estructura semiestructurada

Se construyó `silver_bookings_anidado`, donde cada reserva embebe un array de 
structs con su historial completo de actualizaciones. De las 72,246 reservas, 
47,705 (66%) tienen al menos una actualización registrada, y se identificaron 
casos con hasta 10 actualizaciones sobre una misma reserva.

### Propiedades del lakehouse evidenciadas

- **ACID**: un `UPDATE` sobre 1,377 reservas `pending` vencidas se aplicó de forma 
  atómica, confirmado por el conteo de estados antes/después (`numUpdatedRows: 1377` 
  en las métricas de la operación).
- **Time travel**: `DESCRIBE HISTORY` mostró 2 versiones de `silver_bookings_estado_actual`; 
  se consultó la versión 0 (1,377 reservas `pending`) frente a la versión actual 
  (esas mismas 1,377 como `expired`), confirmando que el historial se preserva.
- **Evolución de esquema**: se agregó la columna `duracion_noches` con `mergeSchema=true` 
  sobre una tabla de 72,246 filas ya escrita, sin migración manual ni pérdida de datos.

### Consultas analíticas — hallazgos principales

1. **Cancelación por país**: los países con mayor tasa de reservas problemáticas 
   (cancelada + vencida) rondan 45–53%, muy por encima del promedio general (~39%), 
   sin un patrón geográfico o continental claro.
2. **Tiempo hasta primera actualización**: promedio de 1,552 horas (~65 días), 
   cifra inusualmente alta que señala una posible inconsistencia en cómo se 
   generaron las fechas de `booking_updates` respecto a `bookings.created_at`.
3. **Modificaciones vs. estado final**: a mayor número de actualizaciones, la tasa 
   de cancelación baja (62.3% → 21.8%) pero la de vencimiento sube (0.37% → 9.12%), 
   sugiriendo que las reservas muy modificadas tienden a quedar sin resolver.
4. **Volumen por mes y continente**: crecimiento sostenido desde finales de 2022, 
   con Asia y África concentrando el mayor volumen en el periodo inicial; se 
   detectó la categoría atípica `"Europe/Asia"` en el campo continente.
5. **Usuarios business vs. individuales**: sin diferencia relevante en monto, 
   duración o número de huéspedes promedio, contradiciendo la hipótesis inicial 
   de que las cuentas empresariales generan reservas de mayor valor.

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*


### Qué funcionó

El modelo lakehouse permitió resolver, con un solo motor y sin infraestructura 
adicional, dos necesidades que normalmente exigirían sistemas separados: 
analítica agregada tipo SQL (JOINs y GROUP BY sobre las 4 tablas) y manejo nativo 
de una relación 1:N como estructura anidada (`silver_bookings_anidado`). Las tres 
propiedades del lakehouse (ACID, time travel, evolución de esquema) se evidenciaron 
sobre datos reales del propio caso, no con ejemplos artificiales, lo cual reforzó 
la justificación de la sección 3: por ejemplo, el `UPDATE` de reservas vencidas 
y su posterior verificación con `DESCRIBE HISTORY` fue el mismo flujo que un 
equipo de operaciones de Wanderbricks necesitaría en producción.

### Qué no funcionó (o requirió corrección)

La consulta 3 inicial arrojó un resultado engañoso (0% de cancelación en toda 
reserva con actualizaciones) por usar `bookings.status` en vez de `status_vigente`; 
esto reveló que el campo `status` original no refleja el estado real de una 
reserva una vez que existen actualizaciones, y justificó *a posteriori* haber 
construido `silver_bookings_estado_actual` en la capa plata. De igual forma, el 
promedio de 1,552 horas hasta la primera actualización (consulta 2) resultó 
demasiado alto para ser operacionalmente realista, lo que sugiere que las fechas 
del dataset no fueron generadas con una relación temporal estricta entre 
`bookings.created_at` y `booking_updates.updated_at` — una limitación del dataset 
de muestra, no del diseño de la base de datos.

### Qué harían distinto si empezaran de nuevo

Se validaría la coherencia temporal entre `created_at` y `updated_at` desde la 
capa bronce (con una regla de calidad de datos explícita), en vez de descubrirlo 
tarde, durante las consultas analíticas. También se normalizaría el campo 
`country` en `users` y `countries` (trim/lower) desde la ingesta a bronce y no 
solo en la capa plata, para evitar tener que repetir esa limpieza en cada consulta 
que involucre países. Finalmente, se documentaría desde el inicio la categoría 
`"Europe/Asia"` en `countries.continent` como una decisión de negocio pendiente 
(¿se trata como Europa, como Asia, o como categoría propia?), en vez de dejarla 
sin resolver hasta encontrarla en los resultados de la consulta 4.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
|Isabela Cuartas  |Descripción de los datos, implementación, consulta analiticas PySpark, resultados |Puntos realizados |
|Juan Camilo Gomez | Contexto y el problema, desiciones de diseño y justificación, propiedades lakehouse, consultas analiticas SQL, conclusiones|Puntos realizados |
| | | |

**Uso de asistentes de IA:** *Claude*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas